In [1]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [2]:
openai = OpenAI()
todos = []
completed = []

In [11]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

def mark_complete(index: int, notes: str) -> str:
    if 1<=index<=len(todos):
        completed[index - 1] = True
    else:
        return "No todos"
    Console().print(notes)
    return get_todo_report()


In [16]:
create_todos(["Buy groceries", "Clean the house", "Finish the project"])

Todo #1: Buy groceries
Todo #2: Clean the house
Todo #3: Finish the project

'Todo #1: Buy groceries\nTodo #2: Clean the house\nTodo #3: Finish the project\n'

In [17]:
mark_complete(2, "I have cleaned the house")

I have cleaned the house

Todo #1: Buy groceries
Todo #2: Clean the house
Todo #3: Finish the project

'Todo #1: Buy groceries\nTodo #2: [green][strike]Clean the house[/strike][/green]\nTodo #3: Finish the project\n'

In [34]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [35]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [36]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [37]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [40]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """
If I invest $5,000 today at an annual interest rate of 7%, compounded monthly,
how much will I have after 10 years?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]


In [41]:
todos, completed = [], []
loop(messages)

Todo #1: Identify given values and required formula for monthly compounding future value
Todo #2: Compute future value FV = P*(1+r/m)^(m*t)
Todo #3: Round to sensible cents and present final amount

Given: principal P = $5,000; nominal annual rate r = 0.07; compounding m = 12/month; time t = 10 years.
Use monthly-compound future value: FV = P\*(1 + r/m)^(m\*t).

Todo #1: Identify given values and required formula for monthly compounding future value
Todo #2: Compute future value FV = P*(1+r/m)^(m*t)
Todo #3: Round to sensible cents and present final amount

Compute: periodic rate = r/m = 0.07/12 = 0.0058333333.
Number of periods = m*t = 12*10 = 120.
Growth factor = (1.0058333333)^120 ≈ 2.009889.
FV ≈ 5000 * 2.009889 = 10049.447.

Todo #1: Identify given values and required formula for monthly compounding future value
Todo #2: Compute future value FV = P*(1+r/m)^(m*t)
Todo #3: Round to sensible cents and present final amount

Round to cents: $10,049.45.

Todo #1: Identify given values and required formula for monthly compounding future value
Todo #2: Compute future value FV = P*(1+r/m)^(m*t)
Todo #3: Round to sensible cents and present final amount

Future value (monthly compounding)

Formula:  FV = P(1 + r/m)^(m·t)

Inputs:
• P = 5,000  
• r = 0.07  
• m = 12  
• t = 10  

Calculation:
• Periodic rate = 0.07/12 = 0.0058333333  
• Periods = 12·10 = 120  
• FV = 5000·(1.0058333333)^120 ≈ 5000·2.009889 ≈ $10,049.45

Answer: After 10 years, you’ll have approximately $10,049.45.